# Function Calling, ReAct и управление циклом агента

> Внимание! Материал ноутбука подходит для работы в Google Colaboratory. Мы не можем гарантировать стабильную работу кода на личных устройствах и на других системах виртуализации.

## Введение

### Постановка проблемы

В первом занятии мы собрали базовый ограниченный цикл: GigaChat предлагал функцию, Python проверял и выполнял её, а `ToolMessage` возвращал результат модели. Теперь сделаем этот цикл управляемым: для того же платежа `P-77` нужно последовательно узнать платёж, аккаунт и правило поддержки, причём следующий шаг зависит от уже полученных данных.

### Что такое ReAct

**ReAct** — подход, в котором языковая модель чередует рассуждение о задаче с действиями во внешней среде. Действие получает новые факты из API, базы данных или другого инструмента, а следующее решение учитывает результат. Подход предложен в работе Yao и соавторов (2022): [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629).

В оригинальной статье цикл записан как `Thought → Action → Observation`, то есть «рассуждение → действие → наблюдение». В приложении нам не нужно запрашивать или сохранять скрытые рассуждения модели. Мы работаем с наблюдаемой частью цикла:

`вопрос → действие (вызов инструмента) → Python-функция → наблюдение (ToolMessage) → следующее действие или итоговый ответ`

- **GigaChat** получает вопрос, историю и результаты инструментов, а затем предлагает следующий вызов или итоговый ответ.
- **Среда выполнения Python** проверяет имя функции и аргументы, выполняет разрешённый инструмент и возвращает результат модели.
- **Наблюдение** — проверенный результат инструмента; именно он даёт модели данные для следующего шага.

Полезно отличать три похожих подхода:

| Подход | Кто задаёт следующий шаг | Когда применять |
|---|---|---|
| Один вызов через Function Calling | модель предлагает одну функцию | достаточно одного обращения к данным |
| Фиксированный рабочий процесс | порядок целиком задан Python-кодом | маршрут заранее известен и не меняется |
| ReAct-агент | модель выбирает действие после нового наблюдения | маршрут зависит от полученных данных |

В этом занятии мы строим **ограниченный ReAct-цикл**. GigaChat работает с тремя инструментами только для чтения: первый шаг можно принудительно выбрать через `tool_choice`, остальные шаги выполняются в режиме `auto`. Среда выполнения сверяет каждое предложение со схемой параметров, наблюдаемым планом и текущим состоянием.

Чтобы цикл не запутался, приложение хранит **состояние** — название текущего шага. Таблица допустимых переходов называется **конечным автоматом состояний**, или FSM. Повторная попытка чтения помогает после временной ошибки, а **кэш** возвращает уже полученный результат точно такого же чтения.

### Цели занятия

- проверить схему параметров, принудительный выбор и ограниченную параллельную группу вызовов;
- понять цикл `вызов инструмента → наблюдение → следующий шаг`;
- добавить наблюдаемый план, критик и одну повторную попытку с уточнением;
- увидеть, как разные наблюдения меняют маршрут агента;
- описать порядок и ветвления небольшой FSM;
- добавить ограниченные повторные попытки с нарастающей задержкой, точный кэш, деградацию и компенсацию;
- экономить обращения к модели с помощью классификатора и ветки без модели;
- запустить ветвящийся ReAct-агент с тремя инструментами на GigaChat-2-Max.

### Предварительные знания

Достаточно пройти первое занятие и понимать, что Python-приложение, а не модель, выполняет функцию из `tool_calls`.

### Что получится в итоге

Агент проверки платежа с инструментами только для чтения. Для двойного списания он проходит маршрут `get_payment → get_account → get_support_policy → итоговый ответ`, а для обычного платежа завершает проверку раньше.

### Понятия цикла агента

| Понятие | Что означает | Для чего нужно |
|---|---|---|
| ReAct | Подход, в котором модель чередует выбор действия и использование результата этого действия. | Решать задачу по шагам, когда следующий шаг зависит от новых данных. |
| Действие (`Action`) | Предложение вызвать инструмент или завершить ответ. | Сделать следующий шаг агента явным. |
| Наблюдение (`Observation`) | Проверенный результат действия, обычно возвращённый в `ToolMessage`. | Обновить состояние фактами перед следующим решением. |
| Состояние | Накопленные сообщения, факты, счётчики и текущий этап процесса. | Не терять контекст между шагами и применять правила переходов. |
| Конечный автомат (`FSM`) | Набор допустимых состояний и переходов между ними. | Запретить действия, которые неуместны на текущем этапе. |
| Политика вызовов | Правила для разрешённых инструментов, аргументов, параллельности и подтверждений. | Проверять предложения модели до выполнения. |
| Наблюдаемый план | Краткий список будущих шагов, сохранённый как обычные данные. | Заранее увидеть намерение агента и проверить его без скрытых рассуждений. |
| Критик (`critic`) | Отдельная проверка плана или действия с решением принять, исправить или отклонить. | Обнаружить ошибку до дорогого или опасного вызова. |
| StateAct | Учебный вариант ReAct, где конечный автомат ограничивает доступные действия. | Соединить гибкость модели с детерминированными правилами процесса. |
| Параллельная группа вызовов | Несколько независимых инструментов, предложенных в одном ответе модели. | Сократить время ожидания, если порядок вызовов не влияет на результат. |

## Карта занятия

Мы разберём одну историю: клиент сообщает о двойном списании `P-77`.

1. **Function Calling:** схемы параметров, принудительный выбор, параллельное чтение и проверка аргументов.
2. **ReAct:** действие, `ToolMessage` и новый шаг после наблюдения.
3. **Расширенный ReAct:** наблюдаемый план, критик и ограниченное исправление вызова.
4. **Состояние:** FSM и инварианты каждого шага.
5. **Ошибки:** повторные попытки, нарастающая задержка, деградация и компенсация.
6. **Экономия:** классификатор, маршрутизация, ветка без модели и кэш.
7. **Практика:** три маршрута с GigaChat и один ответ без модели.


## Установка зависимостей

Устанавим необхожимые библиотеки.

In [1]:
# %pip install -q "deepagents>=0.5,<0.6" "gigachat>=0.2,<0.3" "langchain>=1.3,<2" "langchain-gigachat>=0.5,<0.6" "langgraph>=1.2,<2" "python-dotenv>=1,<2"

#  Добавьте GIGACHAT_CREDENTIALS в панель Colab «Секреты» и разрешите
#  notebook доступ к нему. Затем раскомментируйте следующие три строки.

import os
# from google.colab import userdata
# os.environ["GIGACHAT_CREDENTIALS"] = userdata.get("GIGACHAT_CREDENTIALS")
# Function Calling, ReAct и управление циклом агента

from dotenv import load_dotenv
load_dotenv()
from pprint import pprint

In [2]:
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from typing import NamedTuple

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_gigachat import GigaChat
from pydantic import ValidationError

ENV_FILE = find_dotenv(usecwd=True)
if ENV_FILE:
    load_dotenv(ENV_FILE)
ENV_SOURCE = os.path.basename(ENV_FILE) if ENV_FILE else "Colab Secrets / environment"
GIGACHAT_MODEL = "GigaChat-2-Max"

class PolicyError(RuntimeError):
    """Среда выполнения отклонила шаг, запрещённый текущим состоянием."""

class ToolCallValidationError(PolicyError):
    """Форму ответа модели можно исправить повторным запросом с уточнением."""

class ToolCallAuthorizationError(PolicyError):
    """Инструмента нет в списке разрешённых; повтор не расширяет полномочия."""

class TransientError(RuntimeError):
    """Временная ошибка чтения, которую можно ограниченно повторить."""

class ToolReadResult(NamedTuple):
    value: dict
    attempts: int
    cache_hit: bool
    backoff_delays: tuple[float, ...]

pprint({
    "содержание_занятия": {
        "модель": GIGACHAT_MODEL,
        "источник_настроек": ENV_SOURCE,
        "понятия": [
            "Function Calling", "ReAct", "план", "критик", "FSM",
            "повторные попытки и задержка", "деградация", "компенсация",
            "маршрутизация и кэш",
        ],
    }
})

{'содержание_занятия': {'источник_настроек': '.env',
                        'модель': 'GigaChat-2-Max',
                        'понятия': ['Function Calling',
                                    'ReAct',
                                    'план',
                                    'критик',
                                    'FSM',
                                    'повторные попытки и задержка',
                                    'деградация',
                                    'компенсация',
                                    'маршрутизация и кэш']}}


Занятие охватывает путь от Function Calling до экономии обращений к модели; все сетевые запросы к модели направляются в GigaChat-2-Max.

## 1. Function Calling: схема, принудительный выбор и параллельные вызовы

`@tool` превращает сигнатуру Python-функции в JSON-схему параметров. GigaChat получает имя функции, описание, типы и обязательные поля, а возвращает `tool_calls`. Это предложение, а не команда на исполнение.

Три режима, которые важно различать:

- `tool_choice="auto"` — модель выбирает функцию или отвечает текстом;
- `tool_choice="get_payment"` — ответ API должен содержать вызов указанной функции; это **принудительный выбор**, но Python-функция всё ещё не выполнена;
- несколько `tool_calls` в одном ответе — кандидат на параллельное выполнение.

Даже принудительно выбранный вызов проходит проверку среды выполнения. Мы проверяем список разрешённых инструментов, непустой уникальный `call_id`, закрытую схему аргументов и правила параллелизма. Параллельно разрешены только заранее известные независимые инструменты для чтения. Цепочка `payment → account → policy` остаётся последовательной, потому что аргументы следующего шага появляются из предыдущего наблюдения.

### Инструменты и их схемы параметров

`SimulatedAccountService` имитирует внешний API аккаунтов: первый запрос для каждого `account_id` завершается временной ошибкой `429`, а повторный возвращает данные.

In [3]:
PAYMENTS = {
    "P-77": {
        "payment_id": "P-77", "account_id": "A-1", "amount": 1490,
        "currency": "RUB", "status": "captured", "duplicate_of": "P-76",
    },
    "P-76": {
        "payment_id": "P-76", "account_id": "A-1", "amount": 1490,
        "currency": "RUB", "status": "captured", "duplicate_of": None,
    },
}
ACCOUNTS = {
    "A-1": {"account_id": "A-1", "status": "active", "country": "RU"},
}
SUPPORT_POLICIES = {
    ("duplicate_charge", "RU"): {
        "issue_type": "duplicate_charge", "country": "RU",
        "decision": "manual_refund_review", "sla_hours": 24,
        "required_facts": ["payment_id", "duplicate_of", "account_status"],
    },
}

@tool
def get_payment(payment_id: str) -> dict:
    """Получить проверенные данные платежа по его идентификатору."""
    return dict(PAYMENTS.get(payment_id, {
        "payment_id": payment_id, "status": "not_found",
    }))

class SimulatedAccountService:
    """Имитирует временный сбой внешнего API аккаунтов.

    Первый запрос для каждого account_id завершается ошибкой 429,
    а следующий возвращает данные аккаунта.
    """
    def __init__(self):
        self.calls: dict[str, int] = {}

    def reset(self) -> None:
        self.calls.clear()

    def __call__(self, account_id: str) -> dict:
        self.calls[account_id] = self.calls.get(account_id, 0) + 1
        if self.calls[account_id] == 1:
            raise TransientError("429: служба аккаунтов временно недоступна")
        return dict(ACCOUNTS.get(account_id, {
            "account_id": account_id, "status": "not_found",
        }))

account_service = SimulatedAccountService()

@tool
def get_account(account_id: str) -> dict:
    """Получить проверенный статус аккаунта по его идентификатору."""
    return account_service(account_id)

@tool
def get_support_policy(issue_type: str, country: str) -> dict:
    """Получить правило поддержки только для чтения по типу проблемы и стране."""
    return dict(SUPPORT_POLICIES.get((issue_type, country), {
        "issue_type": issue_type, "country": country,
        "decision": "manual_review", "sla_hours": 48,
    }))

TOOLS = {
    tool.name: tool
    for tool in (get_payment, get_account, get_support_policy)
}

pprint({
    "инструменты": {
        name: {
            key: schema[key]
            for key in ("type", "required", "properties") if key in schema
        }
        for name, tool in TOOLS.items()
        for schema in (tool.args_schema.model_json_schema(),)
    },
    "примеры_платежей": {
        payment_id: get_payment.invoke({"payment_id": payment_id})
        for payment_id in ("P-77", "P-76", "P-404")
    },
    "пример_правила": get_support_policy.invoke({
        "issue_type": "duplicate_charge", "country": "RU",
    }),
    "примечание_о_службе_аккаунтов": (
        "первое чтение account_id приводит к TransientError(429)"
    ),
})

{'инструменты': {'get_account': {'properties': {'account_id': {'title': 'Account '
                                                                        'Id',
                                                               'type': 'string'}},
                                 'required': ['account_id'],
                                 'type': 'object'},
                 'get_payment': {'properties': {'payment_id': {'title': 'Payment '
                                                                        'Id',
                                                               'type': 'string'}},
                                 'required': ['payment_id'],
                                 'type': 'object'},
                 'get_support_policy': {'properties': {'country': {'title': 'Country',
                                                                   'type': 'string'},
                                                       'issue_type': {'title': 'Issue '
                         

Три инструмента только для чтения образуют зависимую цепочку: платёж даёт account_id, а аккаунт — country для поиска правила поддержки.

### Единая проверка ответа модели

Следующая функция принимает тот же словарный формат, который LangChain помещает в `AIMessage.tool_calls`. Она используется дважды: в демонстрации параллельного чтения и внутри критика итогового ReAct-цикла.

In [4]:
PARALLEL_SAFE_TOOLS = {"get_payment", "get_support_policy"}

def validate_function_call_batch(
    calls, *, forced_tool=None, allow_parallel=False, max_parallel_calls=1,
) -> list[dict]:
    """Проверяет tool_calls из ответа модели до исполнения Python-функции."""
    calls = list(calls)
    if not calls:
        raise ToolCallValidationError("ожидается хотя бы один вызов инструмента")
    if (
        not isinstance(max_parallel_calls, int)
        or isinstance(max_parallel_calls, bool)
        or max_parallel_calls < 1
    ):
        raise ValueError("max_parallel_calls должен быть положительным целым числом")

    ids, validated = [], []
    for raw_call in calls:
        if not isinstance(raw_call, dict) or not isinstance(raw_call.get("args"), dict):
            raise ToolCallValidationError("нужен типизированный вызов со словарём аргументов")
        call_id, name, arguments = raw_call.get("id"), raw_call.get("name"), raw_call["args"]
        if not isinstance(call_id, str) or not call_id.strip():
            raise ToolCallValidationError("call_id должен быть непустой строкой")
        if name not in TOOLS:
            raise ToolCallAuthorizationError(
                f"инструмент {name!r} отсутствует в списке разрешённых"
            )

        schema = TOOLS[name].args_schema
        if set(arguments) != set(schema.model_fields):
            raise ToolCallValidationError(
                f"аргументы {name} не совпадают с закрытой схемой"
            )
        try:
            normalized = schema.model_validate(arguments, strict=True).model_dump()
        except ValidationError as error:
            raise ToolCallValidationError(f"аргументы {name} имеют неверные типы") from error
        ids.append(call_id)
        validated.append({"id": call_id, "name": name, "args": normalized})

    if len(set(ids)) != len(ids):
        raise ToolCallValidationError("call_id должны быть уникальны")
    if forced_tool is not None:
        if forced_tool not in TOOLS:
            raise ToolCallAuthorizationError(
                "принудительно выбранного инструмента нет в списке разрешённых"
            )
        if len(validated) != 1 or validated[0]["name"] != forced_tool:
            raise ToolCallValidationError(
                "принудительный выбор требует ровно один указанный инструмент"
            )
    if len(validated) > 1:
        if not allow_parallel or len(validated) > max_parallel_calls:
            raise ToolCallValidationError(
                "параллельная группа запрещена или превышает лимит"
            )
        if any(call["name"] not in PARALLEL_SAFE_TOOLS for call in validated):
            raise ToolCallAuthorizationError(
                "группа содержит зависимый или небезопасный инструмент"
            )
    return validated

pprint({
    "проверка_вызовов": {
        "функция_проверки": validate_function_call_batch.__name__,
        "проверки": [
            "список разрешённых", "закрытая схема", "строгие типы",
            "уникальный call_id", "принудительный выбор",
            "правила параллельного выполнения",
        ],
        "инструменты_для_параллельного_чтения": sorted(PARALLEL_SAFE_TOOLS),
    }
})

{'проверка_вызовов': {'инструменты_для_параллельного_чтения': ['get_payment',
                                                               'get_support_policy'],
                      'проверки': ['список разрешённых',
                                   'закрытая схема',
                                   'строгие типы',
                                   'уникальный call_id',
                                   'принудительный выбор',
                                   'правила параллельного выполнения'],
                      'функция_проверки': 'validate_function_call_batch'}}


Одна функция до исполнения проверяет принудительный выбор, список разрешённых инструментов, закрытую схему, типы, call_id и правила параллельной группы.

### Проверяем принудительный выбор и ошибочные вызовы

Теперь подадим в проверку один корректный принудительный вызов и четыре некорректных варианта. Это модульная проверка границы среды выполнения, а не сохранённые ответы модели: обращение к GigaChat будет выполнено в сетевой части практики.

In [5]:
forced_example = [{
    "id": "forced-1", "name": "get_payment", "args": {"payment_id": "P-77"},
}]
assert validate_function_call_batch(
    forced_example, forced_tool="get_payment",
) == forced_example

function_call_rejections = {}
invalid_function_calls = {
    "неверный_принудительный_выбор": (
        [{"id": "x-1", "name": "get_account", "args": {"account_id": "A-1"}}],
        {"forced_tool": "get_payment"},
    ),
    "неизвестный_инструмент": ([{"id": "x-2", "name": "shell", "args": {}}], {}),
    "неверный_тип": ([{"id": "x-3", "name": "get_payment", "args": {"payment_id": 77}}], {}),
    "лишний_аргумент": (
        [{
            "id": "x-4", "name": "get_payment",
            "args": {"payment_id": "P-77", "debug": True},
        }],
        {},
    ),
}
for case, (calls, options) in invalid_function_calls.items():
    try:
        validate_function_call_batch(calls, **options)
        raise AssertionError(f"пример {case} должен быть отклонён")
    except PolicyError as error:
        function_call_rejections[case] = str(error)

pprint({
    "примеры_вызовов": {
        "принудительный_вызов_принят": forced_example,
        "отклонённые": function_call_rejections,
    }
})

{'примеры_вызовов': {'отклонённые': {'лишний_аргумент': 'аргументы get_payment '
                                                        'не совпадают с '
                                                        'закрытой схемой',
                                     'неверный_принудительный_выбор': 'принудительный '
                                                                      'выбор '
                                                                      'требует '
                                                                      'ровно '
                                                                      'один '
                                                                      'указанный '
                                                                      'инструмент',
                                     'неверный_тип': 'аргументы get_payment '
                                                     'имеют неверные типы',
                                     'неизве

Корректный принудительный вызов принят, а неверный инструмент, тип или лишний аргумент отклонены с понятными причинами.

### Выполняем разрешённую параллельную группу

Для примера одновременно читаем два независимых источника: платёж с уже известным `payment_id` и правило с уже известными `issue_type/country`. Каждый результат возвращается отдельным `ToolMessage` с исходным `tool_call_id`.

В основном расследовании эти вызовы не параллельны: `country` там становится известен только после чтения аккаунта.

In [6]:
def execute_parallel_read_batch(calls: list[dict]) -> list[ToolMessage]:
    """Выполняет только заранее проверенную ограниченную группу чтений."""
    calls = validate_function_call_batch(
        calls, allow_parallel=True, max_parallel_calls=2,
    )
    with ThreadPoolExecutor(max_workers=len(calls)) as pool:
        futures = [
            pool.submit(TOOLS[call["name"]].invoke, call["args"])
            for call in calls
        ]
        results = [future.result() for future in futures]
    return [
        ToolMessage(
            content=json.dumps({"status": "ok", "data": result}, ensure_ascii=False),
            tool_call_id=call["id"], name=call["name"],
        )
        for call, result in zip(calls, results)
    ]

parallel_calls = [
    {"id": "parallel-payment", "name": "get_payment", "args": {"payment_id": "P-76"}},
    {
        "id": "parallel-policy", "name": "get_support_policy",
        "args": {"issue_type": "duplicate_charge", "country": "RU"},
    },
]
parallel_tool_messages = execute_parallel_read_batch(parallel_calls)
assert [message.tool_call_id for message in parallel_tool_messages] == [
    "parallel-payment", "parallel-policy",
]

pprint({
    "параллельная_группа_чтений": [
        {
            "name": message.name,
            "tool_call_id": message.tool_call_id,
            "наблюдение": json.loads(message.content),
        }
        for message in parallel_tool_messages
    ]
})

{'параллельная_группа_чтений': [{'name': 'get_payment',
                                 'tool_call_id': 'parallel-payment',
                                 'наблюдение': {'data': {'account_id': 'A-1',
                                                         'amount': 1490,
                                                         'currency': 'RUB',
                                                         'duplicate_of': None,
                                                         'payment_id': 'P-76',
                                                         'status': 'captured'},
                                                'status': 'ok'}},
                                {'name': 'get_support_policy',
                                 'tool_call_id': 'parallel-policy',
                                 'наблюдение': {'data': {'country': 'RU',
                                                         'decision': 'manual_refund_review',
                                              

Два независимых чтения выполнены параллельно, а каждое наблюдение сохранило собственный tool_call_id из ответа модели.

## 2. Что именно делает ReAct-агент

Один Function Calling отвечает на вопрос «какую Python-функцию вызвать сейчас?». ReAct повторяет этот механизм после каждого нового наблюдения.

В нашем примере маршрут выглядит так:

`вопрос → get_payment → наблюдение → следующий инструмент или ответ`

Если найдено двойное списание, маршрут продолжается:

`get_account → наблюдение → get_support_policy → наблюдение → ответ`

Если `duplicate_of` отсутствует, наблюдение меняет маршрут: агент не читает аккаунт и правило без необходимости, а сразу готовит ответ.

GigaChat формирует вызов инструмента и итоговый текст. Python-приложение контролирует порядок, аргументы, повторные попытки, кэш и фактический вызов функций. Скрытые рассуждения модели сохранять не требуется: для отладки достаточно видеть состояние, вызовы инструментов и наблюдения.

### Что именно возвращается модели

Результат Python-функции помещается в `ToolMessage`. Его `tool_call_id` должен совпадать с идентификатором исходного вызова — так история связывает запрос функции с соответствующим наблюдением.

Следующая ячейка собирает проверенное наблюдение из `get_payment` и создаёт сообщение той же вспомогательной функцией, которую затем использует итоговый цикл.

In [7]:
def make_tool_message(call: dict, observation: dict) -> ToolMessage:
    """Связывает проверенное наблюдение с исходным tool_call_id модели."""
    return ToolMessage(
        content=json.dumps(observation, ensure_ascii=False),
        tool_call_id=call["id"],
        name=call["name"],
    )

example_call = {
    "id": "example-payment-call",
    "name": "get_payment",
    "args": {"payment_id": "P-77"},
}
example_observation = {
    "status": "ok",
    "data": get_payment.invoke(example_call["args"]),
    "attempts": 1,
    "cache_hit": False,
}
example_tool_message = make_tool_message(example_call, example_observation)
assert example_tool_message.tool_call_id == example_call["id"]
assert json.loads(example_tool_message.content) == example_observation

pprint({
    "контракт_tool_message": {
        "name": example_tool_message.name,
        "tool_call_id": example_tool_message.tool_call_id,
        "прочитанное_наблюдение": json.loads(example_tool_message.content),
    }
})

{'контракт_tool_message': {'name': 'get_payment',
                           'tool_call_id': 'example-payment-call',
                           'прочитанное_наблюдение': {'attempts': 1,
                                                      'cache_hit': False,
                                                      'data': {'account_id': 'A-1',
                                                               'amount': 1490,
                                                               'currency': 'RUB',
                                                               'duplicate_of': 'P-76',
                                                               'payment_id': 'P-77',
                                                               'status': 'captured'},
                                                      'status': 'ok'}}}


`ToolMessage` сохраняет проверенный результат и исходный `tool_call_id`; та же функция используется на каждом шаге итогового цикла.

## 3. Расширенный ReAct: план, критик и повтор с уточнением

Перед первым вызовом инструмента среда выполнения строит **наблюдаемый план фактов**: что нужно проверить и при каком условии. Это не скрытая цепочка рассуждений модели, а обычная структура данных, которую можно вывести, проверить и сократить после наблюдения.

После каждого ответа модели работает **критик** — проверка по явным правилам перед Python-вызовом:

- корректный вызов получает `approved`;
- неверные аргументы или преждевременный разрешённый инструмент получают `repair` — GigaChat видит конкретное уточнение и получает ещё одну попытку;
- неизвестный инструмент получает конечный `policy_denied`: повтор не расширит список разрешённых инструментов.

Отклонённый `AIMessage` с незакрытым вызовом инструмента не добавляется в историю сообщений. Среда выполнения повторяет запрос с описанием ошибки проверки и сохраняет только окончательно принятый вызов вместе с парным `ToolMessage`.

### Наблюдаемый план до первого вызова

In [8]:
PLAN_TEMPLATE = (
    {"fact": "payment", "tool": "get_payment", "condition": "always"},
    {"fact": "account", "tool": "get_account", "condition": "duplicate_of is not None"},
    {
        "fact": "support_policy", "tool": "get_support_policy",
        "condition": "account.status == active",
    },
)

def plan_before_calls(payment_id: str) -> list[dict]:
    """Возвращает наблюдаемый план до обращений к модели и инструментам."""
    return [
        {"position": index, "payment_id": payment_id, **step}
        for index, step in enumerate(PLAN_TEMPLATE, start=1)
    ]

FACT_BY_STATE = {
    "проверка_платежа": "payment",
    "проверка_аккаунта": "account",
    "проверка_правил": "support_policy",
}

def remaining_plan(state: str, plan: list[dict]) -> list[dict]:
    """Показывает ещё не выполненную часть плана для текущего состояния."""
    next_fact = FACT_BY_STATE.get(state)
    if next_fact is None:
        return []
    start = next(index for index, step in enumerate(plan) if step["fact"] == next_fact)
    return plan[start:]

plan_example = plan_before_calls("P-77")
assert [step["fact"] for step in remaining_plan("проверка_аккаунта", plan_example)] == [
    "account", "support_policy",
]

pprint({
    "наблюдаемый_план": {
        "начальный": plan_example,
        "после_платежа": remaining_plan("проверка_аккаунта", plan_example),
        "после_аккаунта": remaining_plan("проверка_правил", plan_example),
    }
})

{'наблюдаемый_план': {'начальный': [{'condition': 'always',
                                     'fact': 'payment',
                                     'payment_id': 'P-77',
                                     'position': 1,
                                     'tool': 'get_payment'},
                                    {'condition': 'duplicate_of is not None',
                                     'fact': 'account',
                                     'payment_id': 'P-77',
                                     'position': 2,
                                     'tool': 'get_account'},
                                    {'condition': 'account.status == active',
                                     'fact': 'support_policy',
                                     'payment_id': 'P-77',
                                     'position': 3,
                                     'tool': 'get_support_policy'}],
                      'после_аккаунта': [{'condition': 'account.status == '
         

План существует как обычные данные до обращений к модели и последовательно сокращается от трёх возможных проверок к одной.

### Критик: `approved`, `repair` или `policy_denied`

In [9]:
class CriticReview(NamedTuple):
    outcome: str
    feedback: str | None
    call: dict | None

def critic_review(
    calls, *, expected_tool: str, expected_arguments: dict, forced_tool=None,
) -> CriticReview:
    """Разделяет approved, исправимый repair и конечный policy_denied."""
    try:
        validated = validate_function_call_batch(calls, forced_tool=forced_tool)
    except ToolCallValidationError as error:
        return CriticReview("repair", str(error), None)
    except ToolCallAuthorizationError as error:
        return CriticReview("policy_denied", str(error), None)
    call = validated[0]
    if call["name"] != expected_tool:
        return CriticReview("repair", f"сейчас ожидается {expected_tool}", None)
    if call["args"] != expected_arguments:
        return CriticReview("repair", f"ожидались аргументы {expected_arguments}", None)
    return CriticReview("approved", None, call)

def clarification_for(review: CriticReview) -> HumanMessage:
    """Даёт модели замечание критика без отклонённого AIMessage в истории."""
    return HumanMessage(content=(
        f"Предыдущее предложение отклонено средой выполнения: {review.feedback}. "
        "Исправь только вызов инструмента и используй проверенные идентификаторы "
        "из текущего состояния."
    ))

critic_examples = {
    "approved": critic_review(
        forced_example, expected_tool="get_payment",
        expected_arguments={"payment_id": "P-77"},
    ),
    "repair": critic_review(
        [{"id": "critic-2", "name": "get_payment", "args": {"payment_id": "P-76"}}],
        expected_tool="get_payment", expected_arguments={"payment_id": "P-77"},
    ),
    "policy_denied": critic_review(
        [{"id": "critic-3", "name": "shell", "args": {}}],
        expected_tool="get_payment", expected_arguments={"payment_id": "P-77"},
    ),
}
assert [review.outcome for review in critic_examples.values()] == [
    "approved", "repair", "policy_denied",
]

pprint({
    "критик": {
        "решения_критика": {
            name: review._asdict() for name, review in critic_examples.items()
        },
        "уточнение_для_исправления": clarification_for(critic_examples["repair"]).content,
    }
})

{'критик': {'решения_критика': {'approved': {'call': {'args': {'payment_id': 'P-77'},
                                                      'id': 'forced-1',
                                                      'name': 'get_payment'},
                                             'feedback': None,
                                             'outcome': 'approved'},
                                'policy_denied': {'call': None,
                                                  'feedback': 'инструмент '
                                                              "'shell' "
                                                              'отсутствует в '
                                                              'списке '
                                                              'разрешённых',
                                                  'outcome': 'policy_denied'},
                                'repair': {'call': None,
                                           'feedback': 

Критик разделяет approved, исправимый repair и конечный policy_denied, не сохраняя скрытые рассуждения модели.

## 4. Состояние и FSM: управляем маршрутом

**Состояние** — короткая метка текущего шага. Она не является ответом модели и меняется только Python-кодом.

Наш FSM содержит пять состояний:

- `проверка_платежа` — разрешён только `get_payment`;
- `проверка_аккаунта` — разрешён только `get_account`;
- `проверка_правил` — разрешён только `get_support_policy`;
- `подготовка_ответа` — инструменты больше не нужны;
- `завершено` — цикл остановлен.

Две таблицы связывают состояние с циклом:

- `EXPECTED_TOOL_BY_STATE` говорит, какой инструмент разрешён сейчас;
- `event_from_observation` превращает данные инструмента в событие FSM.

GigaChat работает в режиме `auto` и сам предлагает инструмент. FSM не выбирает вместо модели, а проверяет её предложение. Важное отличие от фиксированного рабочего процесса: следующее состояние зависит не только от имени инструмента, но и от данных наблюдения.

In [10]:
STATE_TRANSITIONS = {
    "проверка_платежа": {
        "дубликат_найден": "проверка_аккаунта",
        "дубликат_не_найден": "подготовка_ответа",
        "платеж_не_найден": "подготовка_ответа",
    },
    "проверка_аккаунта": {
        "аккаунт_активен": "проверка_правил",
        "аккаунт_недоступен": "подготовка_ответа",
    },
    "проверка_правил": {"правило_получено": "подготовка_ответа"},
    "подготовка_ответа": {"ответ_готов": "завершено"},
    "завершено": {},
}
EXPECTED_TOOL_BY_STATE = {
    "проверка_платежа": "get_payment",
    "проверка_аккаунта": "get_account",
    "проверка_правил": "get_support_policy",
}

pprint({
    "описание_fsm": {
        "переходы": STATE_TRANSITIONS,
        "разрешённый_инструмент_по_состоянию": EXPECTED_TOOL_BY_STATE,
    }
})

{'описание_fsm': {'переходы': {'завершено': {},
                               'подготовка_ответа': {'ответ_готов': 'завершено'},
                               'проверка_аккаунта': {'аккаунт_активен': 'проверка_правил',
                                                     'аккаунт_недоступен': 'подготовка_ответа'},
                               'проверка_платежа': {'дубликат_найден': 'проверка_аккаунта',
                                                    'дубликат_не_найден': 'подготовка_ответа',
                                                    'платеж_не_найден': 'подготовка_ответа'},
                               'проверка_правил': {'правило_получено': 'подготовка_ответа'}},
                  'разрешённый_инструмент_по_состоянию': {'проверка_аккаунта': 'get_account',
                                                          'проверка_платежа': 'get_payment',
                                                          'проверка_правил': 'get_support_policy'}}}


Таблица заранее показывает обе ветки: только три рабочих состояния разрешают инструменты, а подготовка ответа и завершение не запускают функции.

In [11]:
def transition_state(state: str, event: str) -> str:
    transitions = STATE_TRANSITIONS.get(state)
    if transitions is None or event not in transitions:
        raise PolicyError(f"переход {state} --{event}--> ? запрещён")
    return transitions[event]

state_path = ["проверка_платежа"]
for event in (
    "дубликат_найден", "аккаунт_активен",
    "правило_получено", "ответ_готов",
):
    state_path.append(transition_state(state_path[-1], event))
assert state_path == [
    "проверка_платежа", "проверка_аккаунта", "проверка_правил",
    "подготовка_ответа", "завершено",
]
short_path = ["проверка_платежа"]
for event in ("дубликат_не_найден", "ответ_готов"):
    short_path.append(transition_state(short_path[-1], event))
assert short_path == ["проверка_платежа", "подготовка_ответа", "завершено"]
try:
    transition_state("проверка_платежа", "аккаунт_активен")
    raise AssertionError("нельзя пропустить проверку платежа")
except PolicyError as error:
    blocked_transition = str(error)

pprint({
    "fsm": {
        "путь_дубликата": state_path,
        "путь_обычного_платежа": short_path,
        "инструмент_по_состоянию": EXPECTED_TOOL_BY_STATE,
        "запрещённый_переход": blocked_transition,
    }
})

{'fsm': {'запрещённый_переход': 'переход проверка_платежа --аккаунт_активен--> '
                                '? запрещён',
         'инструмент_по_состоянию': {'проверка_аккаунта': 'get_account',
                                     'проверка_платежа': 'get_payment',
                                     'проверка_правил': 'get_support_policy'},
         'путь_дубликата': ['проверка_платежа',
                            'проверка_аккаунта',
                            'проверка_правил',
                            'подготовка_ответа',
                            'завершено'],
         'путь_обычного_платежа': ['проверка_платежа',
                                   'подготовка_ответа',
                                   'завершено']}}


FSM содержит длинную и короткую ветки: одна проверяет аккаунт и правило, другая сразу переходит к ответу; недопустимый переход блокируется.

### Как наблюдение меняет маршрут

После `get_payment` одного события «платёж получен» недостаточно. Среда выполнения должна различить как минимум три результата: найден дубликат, дубликат не найден и платёж не найден.

`event_from_observation` выполняет эту интерпретацию по явным правилам. Модель видит наблюдение и предлагает следующий шаг, но именно Python-код превращает проверенные данные в событие FSM.

In [12]:
def event_from_observation(state: str, tool_name: str, data: dict) -> str:
    """Преобразует проверенный результат инструмента в событие состояния."""
    if EXPECTED_TOOL_BY_STATE.get(state) != tool_name:
        raise PolicyError(f"инструмент {tool_name} не соответствует состоянию {state}")

    if state == "проверка_платежа":
        if data.get("status") == "not_found":
            return "платеж_не_найден"
        return "дубликат_найден" if data.get("duplicate_of") else "дубликат_не_найден"
    if state == "проверка_аккаунта":
        return "аккаунт_активен" if data.get("status") == "active" else "аккаунт_недоступен"
    if state == "проверка_правил":
        return "правило_получено"
    raise PolicyError(f"для состояния {state} нет правила обработки наблюдения")

observation_routes = {
    payment_id: event_from_observation(
        "проверка_платежа", "get_payment",
        get_payment.invoke({"payment_id": payment_id}),
    )
    for payment_id in ("P-77", "P-76", "P-404")
}
assert observation_routes == {
    "P-77": "дубликат_найден",
    "P-76": "дубликат_не_найден",
    "P-404": "платеж_не_найден",
}
try:
    event_from_observation(
        "проверка_платежа", "get_account", {"status": "active"},
    )
    raise AssertionError("инструмент другого состояния должен быть отклонён")
except PolicyError as error:
    wrong_tool_reason = str(error)

pprint({
    "маршрутизация_по_наблюдению": observation_routes,
    "отклонённый_инструмент": wrong_tool_reason,
    "смысл": "один инструмент даёт разные события и следующие состояния",
})

{'маршрутизация_по_наблюдению': {'P-404': 'платеж_не_найден',
                                 'P-76': 'дубликат_не_найден',
                                 'P-77': 'дубликат_найден'},
 'отклонённый_инструмент': 'инструмент get_account не соответствует состоянию '
                           'проверка_платежа',
 'смысл': 'один инструмент даёт разные события и следующие состояния'}


Один get_payment даёт разные события для дубликата, обычного и неизвестного платежа — маршрут выбирается по данным наблюдения.

## 5. Ошибки: повторные попытки, нарастающая задержка и кэш

`read_with_retry_cache` делает две вещи:

1. повторяет вызов только при временной ошибке и не превышает `max_attempts`;
2. сохраняет успешный результат по ключу «имя инструмента + аргументы».

Перед повтором используется **нарастающая задержка** (backoff). Параметр `sleep_fn` нужен для проверяемости: в упражнении список записывает рассчитанную задержку без реального ожидания, а итоговый цикл использует обычный `time.sleep`.

Если точно такое же чтение повторится, служба данных не вызывается: результат приходит из кэша с `attempts=0`. Функции с побочными эффектами таким способом кэшировать нельзя.

Кэш передаётся среде выполнения извне и живёт только в рамках сеанса. В сетевой части практики два запуска одного расследования используют общий кэш: первый заполнит запись об аккаунте, а повтор покажет `cache_hit=True` без нового обращения к внешнему API.

### Кэшируем только успешные чтения

Нельзя считать успешным любой словарь, который служба данных вернула без исключения. Среда выполнения кэширует только явно успешные статусы; `degraded`, `temporarily_unavailable`, `partial`, `error` и неизвестные статусы каждый раз перепроверяются.

In [13]:
CACHEABLE_READ_STATUSES = {
    "ok", "active", "captured", "operational",
}

def is_successful_read(value) -> bool:
    """Кэширует только результат с явно успешным значением поля status."""
    status = (
        value.get("status")
        if isinstance(value, dict)
        else getattr(value, "status", None)
    )
    return status in CACHEABLE_READ_STATUSES

cache_policy_examples = {
    status: is_successful_read({"status": status})
    for status in (
        "active", "degraded", "temporarily_unavailable",
        "partial", "error", "unexpected",
    )
}
assert cache_policy_examples == {
    "active": True,
    "degraded": False,
    "temporarily_unavailable": False,
    "partial": False,
    "error": False,
    "unexpected": False,
}

pprint({
    "кэшируются_только_успешные": cache_policy_examples,
    "кэшируемые_статусы": sorted(CACHEABLE_READ_STATUSES),
})

{'кэшируемые_статусы': ['active', 'captured', 'ok', 'operational'],
 'кэшируются_только_успешные': {'active': True,
                                'degraded': False,
                                'error': False,
                                'partial': False,
                                'temporarily_unavailable': False,
                                'unexpected': False}}


Кэш принимает только явно успешные статусы; деградация, временная недоступность и неизвестные результаты не становятся устаревшим ответом следующего запуска.

In [15]:
def read_with_retry_cache(
    tool_name, arguments, execute, cache, *, max_attempts=2,
    base_delay=0.05, sleep_fn=time.sleep,
):
    """Выполняет безопасное чтение с точным кэшем и ограниченными повторами.
    tool_name и arguments образуют ключ; execute обращается к службе.
    cache хранит только успешные чтения и передаётся снаружи.
    max_attempts — общее число обращений к службе, включая первую попытку.
    sleep_fn позволяет тесту записывать задержки вместо реального ожидания.
    """
    # bool — подкласс int, поэтому отклоняем его отдельно.
    if not isinstance(max_attempts, int) or isinstance(max_attempts, bool) or max_attempts < 1:
        raise ValueError("max_attempts должен быть положительным целым числом")

    # Готовый канонический ключ делает cache hit независимым от порядка полей.
    cache_key = json.dumps(
        {"tool": tool_name, "arguments": arguments},
        sort_keys=True, ensure_ascii=False,
    )
    if cache_key in cache:
        return ToolReadResult(cache[cache_key], 0, True, ())

    delays = []
    for attempt in range(1, max_attempts + 1):
        try:
            value = execute(arguments)
            if is_successful_read(value):
                cache[cache_key] = value
            return ToolReadResult(value, attempt, False, tuple(delays))
        except TransientError:
            if attempt == max_attempts:
                raise
            # Нарастающая задержка уже дана: 0.05, 0.1, 0.2 и так далее.
            delay = base_delay * (2 ** (attempt - 1))
            delays.append(delay)
            sleep_fn(delay)

    raise AssertionError("недостижимая ветка")

# Сценарий 1: временная ошибка → успешный повтор → точный cache hit.
account_service.reset()
read_cache, recorded_delays = {}, []

# list.append вместо sleep мгновенно записывает выбранные алгоритмом задержки.
first_read = read_with_retry_cache(
    "get_account", {"account_id": "A-1"},
    lambda args: get_account.invoke(args), read_cache,
    sleep_fn=recorded_delays.append,
)

# Тот же инструмент и те же аргументы должны дать cache hit.
second_read = read_with_retry_cache(
    "get_account", {"account_id": "A-1"},
    lambda args: get_account.invoke(args), read_cache,
    sleep_fn=recorded_delays.append,
)

# Первая операция потребовала повтора, вторая не вызвала службу.
assert first_read.attempts == 2 and first_read.cache_hit is False
assert first_read.backoff_delays == (0.05,)
assert second_read.attempts == 0 and second_read.cache_hit is True
assert account_service.calls == {"A-1": 2}
assert recorded_delays == [0.05]

# Сценарий 2: деградация возвращается вызывающему коду,
# но не считается успешным чтением и не загрязняет кэш.
degraded_service_calls, degraded_cache = [], {}
def return_degraded(arguments):
    degraded_service_calls.append(dict(arguments))
    return {"status": "degraded", "route": "manual_review"}

degraded_first = read_with_retry_cache(
    "get_account", {"account_id": "B-2"},
    return_degraded, degraded_cache,
)
degraded_second = read_with_retry_cache(
    "get_account", {"account_id": "B-2"},
    return_degraded, degraded_cache,
)
assert len(degraded_service_calls) == 2
assert degraded_cache == {}
assert not degraded_first.cache_hit and not degraded_second.cache_hit

# Сценарий 3: неверный лимит отклоняется; bool в Python является подклассом int.
invalid_retry_limits = {}
for invalid_limit in (True, 0, 1.5):
    try:
        read_with_retry_cache(
            "get_payment", {"payment_id": "P-76"},
            lambda args: get_payment.invoke(args), {},
            max_attempts=invalid_limit,
        )
        raise AssertionError("недопустимый max_attempts должен быть отклонён")
    except ValueError as error:
        invalid_retry_limits[repr(invalid_limit)] = str(error)
assert len(invalid_retry_limits) == 3

# Показываем попытки, задержки и cache hit, а не только прохождение assert.
pprint({
    "повторы_и_кэш": {
        "первое_чтение": first_read._asdict(),
        "повтор_того_же_чтения": second_read._asdict(),
        "обращения_к_службе": account_service.calls,
        "задержки": recorded_delays,
        "деградация_не_кэшируется": {
            "обращения_к_службе": len(degraded_service_calls),
            "записи_кэша": len(degraded_cache),
        },
        "недопустимые_лимиты_отклонены": invalid_retry_limits,
    }
})

{'повторы_и_кэш': {'деградация_не_кэшируется': {'записи_кэша': 0,
                                                'обращения_к_службе': 2},
                   'задержки': [0.05],
                   'недопустимые_лимиты_отклонены': {'0': 'max_attempts должен '
                                                          'быть положительным '
                                                          'целым числом',
                                                     '1.5': 'max_attempts '
                                                            'должен быть '
                                                            'положительным '
                                                            'целым числом',
                                                     'True': 'max_attempts '
                                                             'должен быть '
                                                             'положительным '
                                                   

Временная ошибка 429 потребовала две попытки, а точный повтор успешного чтения вернулся из кэша без нового обращения к службе.

### Деградация и компенсация — разные ответы на ошибку

- **Деградация** применяется, когда источник только для чтения остался недоступен после повторных попыток. Агент завершает обработку с ограниченным результатом `manual_review`, не выдумывает отсутствующие факты и не создаёт побочный эффект.
- **Компенсация** применяется, когда побочный эффект уже частично начался. В примере цепочка операций (saga) резервирует возврат, получает ошибку следующего шага и отдельным действием отменяет резерв.

Компенсация не является «обратной повторной попыткой»: это явно спроектированная бизнес-операция. Итоговый агент остаётся доступным модели только для чтения; цепочка операций показана отдельно и не передаётся GigaChat как инструмент.

In [16]:
class UnavailableAccountService:
    """Имитирует API аккаунтов, недоступное во время всех попыток."""
    def __init__(self):
        self.calls = 0

    def __call__(self, _arguments):
        self.calls += 1
        raise TransientError("429: служба всё ещё недоступна")

class ResilientReadOutcome(NamedTuple):
    status: str
    read: ToolReadResult
    route: str
    error: str | None

def read_or_degrade(
    tool_name, arguments, execute, cache, *, max_attempts=2,
    base_delay=0.05, sleep_fn=time.sleep,
) -> ResilientReadOutcome:
    """После исчерпания попыток возвращает типизированную деградацию."""
    recorded_delays = []

    def sleep_and_record(delay):
        recorded_delays.append(delay)
        sleep_fn(delay)

    try:
        read = read_with_retry_cache(
            tool_name, arguments, execute, cache,
            max_attempts=max_attempts, base_delay=base_delay,
            sleep_fn=sleep_and_record,
        )
        return ResilientReadOutcome("ok", read, "continue", None)
    except TransientError as error:
        fallback = {
            **arguments, "status": "temporarily_unavailable", "checked": False,
        }
        read = ToolReadResult(
            fallback, max_attempts, False, tuple(recorded_delays),
        )
        return ResilientReadOutcome(
            "degraded", read, "manual_review", str(error),
        )

if "read_with_retry_cache" in globals():
    unavailable_service = UnavailableAccountService()
    degraded_read_example = read_or_degrade(
        "get_account", {"account_id": "B-2"}, unavailable_service, {},
        max_attempts=2, sleep_fn=lambda _delay: None,
    )
    assert degraded_read_example.status == "degraded"
    assert degraded_read_example.route == "manual_review"
    assert unavailable_service.calls == 2
else:
    degraded_read_example = {
        "status": "доступно после реализации повторных попыток и кэша"
    }

pprint({
    "деградация_после_повторов": (
        degraded_read_example._asdict()
        if hasattr(degraded_read_example, "_asdict")
        else degraded_read_example
    ),
})

{'деградация_после_повторов': {'error': '429: служба всё ещё недоступна',
                               'read': ToolReadResult(value={'account_id': 'B-2', 'status': 'temporarily_unavailable', 'checked': False}, attempts=2, cache_hit=False, backoff_delays=(0.05,)),
                               'route': 'manual_review',
                               'status': 'degraded'}}


После двух неудачных попыток среда выполнения не зацикливается: возвращает типизированную деградацию, сохраняет причину и направляет случай на ручную проверку.

### Зачем нужен следующий пример

До сих пор агент только читал данные, поэтому после ошибки было достаточно завершить проверку безопасным результатом. Но бизнес-операция может состоять из нескольких шагов и успеть частично изменить внешнюю систему.

Следующий блок моделирует цепочку возврата (**saga**):

1. система резервирует возврат;
2. затем пытается уведомить клиента;
3. если уведомление не удалось, отдельная компенсирующая операция отменяет резерв.

Повторять всю цепочку с начала опасно: можно создать второй резерв. Поэтому `run_refund_saga` показывает не retry, а явную отмену уже выполненного шага. В примере `fail_notification` воспроизводит сбой, `events` хранит порядок действий, а `net_effect=None` означает, что после компенсации запланированного возврата не осталось.

Это отдельная учебная демонстрация надёжности. Она не является инструментом GigaChat и не входит в основной ReAct-цикл.

In [ ]:
def run_refund_saga(arguments: dict, *, fail_notification: bool) -> dict:
    """Компенсирует резерв, если следующий шаг цепочки завершился ошибкой."""
    events = [{"event": "возврат_зарезервирован", "arguments": dict(arguments)}]
    if fail_notification:
        events.extend([
            {"event": "уведомление_не_отправлено"},
            {"event": "резерв_возврата_отменён"},
        ])
        return {"status": "compensated", "events": events, "net_effect": None}
    events.append({"event": "клиент_уведомлён"})
    return {
        "status": "committed", "events": events,
        "net_effect": {"refund_scheduled": dict(arguments)},
    }

saga_arguments = {"payment_id": "P-77", "amount": 1490, "currency": "RUB"}
compensation_scenarios = {
    "успех": run_refund_saga(saga_arguments, fail_notification=False),
    "ошибка_уведомления": run_refund_saga(saga_arguments, fail_notification=True),
}
assert compensation_scenarios["успех"]["status"] == "committed"
assert compensation_scenarios["ошибка_уведомления"]["status"] == "compensated"
assert compensation_scenarios["ошибка_уведомления"]["net_effect"] is None

pprint({
    "компенсация": compensation_scenarios,
    "исполнительный_инструмент_доступен_модели": False,
})

Успешная цепочка фиксирует запланированный возврат, а ошибка уведомления запускает компенсирующее действие и оставляет `net_effect=None`.

## 6. Экономия обращений к модели: классификатор и маршрутизация

До создания клиента GigaChat дешёвый локальный классификатор выбирает маршрут:

- `deterministic` — точный FAQ отвечает Python-кодом, `model_calls=0`, то есть без обращения к модели;
- `agent` — наблюдения действительно меняют маршрут, поэтому нужен ReAct;
- `unsupported` — приложение просит уточнение без бессодержательного обращения к модели.

Внутри агентной ветки экономию продолжают кэш только успешных чтений и сокращающийся план. Если приложение уже знает точный ответ или имеет свежий результат точно такого же чтения, новое обращение к модели или службе данных не нужно.

In [17]:
FAQ_ANSWERS = {
    "как сменить пароль?": "Откройте настройки профиля → Безопасность → Сменить пароль.",
}

class RouteDecision(NamedTuple):
    route: str
    label: str
    answer: str | None

def classify_request(message: str) -> str:
    """Дешёвый классификатор выполняется до создания клиента GigaChat."""
    normalized = " ".join(message.casefold().split())
    if normalized in FAQ_ANSWERS:
        return "точное_faq"
    if re.search(r"\bP-\d+\b", message, re.IGNORECASE):
        return "платёжное_расследование"
    return "нужно_уточнение"

def route_request(message: str) -> RouteDecision:
    label = classify_request(message)
    normalized = " ".join(message.casefold().split())
    if label == "точное_faq":
        return RouteDecision("deterministic", label, FAQ_ANSWERS[normalized])
    if label == "платёжное_расследование":
        return RouteDecision("agent", label, None)
    return RouteDecision(
        "unsupported", label,
        "Уточните тип проблемы и идентификатор платежа вида P-123.",
    )

routing_examples = {
    message: route_request(message)
    for message in (
        "Как сменить пароль?",
        "Проверь двойное списание P-77",
        "У меня проблема с оплатой",
    )
}
assert [decision.route for decision in routing_examples.values()] == [
    "deterministic", "agent", "unsupported",
]

pprint({
    "маршрутизация": {
        message: decision._asdict()
        for message, decision in routing_examples.items()
    }
})

{'маршрутизация': {'Как сменить пароль?': {'answer': 'Откройте настройки '
                                                     'профиля → Безопасность → '
                                                     'Сменить пароль.',
                                           'label': 'точное_faq',
                                           'route': 'deterministic'},
                   'Проверь двойное списание P-77': {'answer': None,
                                                     'label': 'платёжное_расследование',
                                                     'route': 'agent'},
                   'У меня проблема с оплатой': {'answer': 'Уточните тип '
                                                           'проблемы и '
                                                           'идентификатор '
                                                           'платежа вида '
                                                           'P-123.',
                                      

Классификатор до создания модели различает точный FAQ, расследование агента и безопасное уточнение; две ветки обходятся без GigaChat.

## 7. Практика: собираем агента на GigaChat

Теперь соединим функции из предыдущих ячеек с GigaChat-2-Max. В длинном сценарии первый шаг принудительно выбирает `get_payment`, остальные работают в режиме `auto`; критик и FSM проверяют каждое предложение. После Python-вызова среда выполнения возвращает наблюдение модели.

Переменные загружаются из ближайшего `.env`: поиск начинается в текущем каталоге и продолжается в родительских. Никаких дополнительных флагов запуска нет.

### Подготовка к запуску

Сначала подключим GigaChat и научимся получать аргументы из уже проверенных фактов. Затем отдельная вспомогательная функция запустит критика и даст модели одну попытку исправления. Все эти функции вызывает итоговая среда выполнения.

In [18]:
def build_gigachat() -> GigaChat:
    """Создаёт GigaChat-2-Max из переменных окружения или файла .env."""
    credentials = os.getenv("GIGACHAT_CREDENTIALS")
    if not credentials:
        raise RuntimeError(
            "задайте GIGACHAT_CREDENTIALS в корневом .env или в Colab Secrets"
        )
    kwargs = {
        "credentials": credentials,
        "scope": os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_B2B"),
        "model": GIGACHAT_MODEL,
        "verify_ssl_certs": False,
    }
    if os.getenv("GIGACHAT_CA_BUNDLE_FILE"):
        kwargs["ca_bundle_file"] = os.environ["GIGACHAT_CA_BUNDLE_FILE"]
    if os.getenv("GIGACHAT_BASE_URL"):
        kwargs["base_url"] = os.environ["GIGACHAT_BASE_URL"]
    return GigaChat(**kwargs)

def expected_arguments_for(tool_name: str, payment_id: str, facts: dict) -> dict:
    if tool_name == "get_payment":
        return {"payment_id": payment_id}
    if tool_name == "get_account":
        payment = facts.get("get_payment")
        if not payment or payment.get("status") == "not_found":
            raise PolicyError("платёж не найден; получить account_id невозможно")
        return {"account_id": payment["account_id"]}
    if tool_name == "get_support_policy":
        account = facts.get("get_account")
        if not account or account.get("status") == "not_found":
            raise PolicyError("аккаунт не найден; определить country невозможно")
        return {"issue_type": "duplicate_charge", "country": account["country"]}
    raise PolicyError(f"неизвестный инструмент {tool_name}")

try:
    expected_arguments_for(
        "get_account", "P-404", {"get_payment": {"status": "not_found"}},
    )
    raise AssertionError("без account_id нельзя переходить к get_account")
except PolicyError as error:
    unknown_payment_reason = str(error)

pprint({
    "подготовка_среды_выполнения": {
        "подключение_модели": build_gigachat.__name__,
        "общая_проверка": validate_function_call_batch.__name__,
        "аргументы_из_состояния": expected_arguments_for.__name__,
        "неизвестный_платёж": unknown_payment_reason,
    }
})

{'подготовка_среды_выполнения': {'аргументы_из_состояния': 'expected_arguments_for',
                                 'неизвестный_платёж': 'платёж не найден; '
                                                       'получить account_id '
                                                       'невозможно',
                                 'общая_проверка': 'validate_function_call_batch',
                                 'подключение_модели': 'build_gigachat'}}


Подготовка среды выполнения отделяет создание GigaChat от получения зависимых аргументов из уже проверенных фактов.

In [19]:
def request_validated_action(
    model, messages, step_prompt, *, expected_tool, expected_arguments,
    forced_tool=None, max_repairs=1,
):
    """Получает предложение модели и один раз уточняет исправимую ошибку."""
    feedback_messages, critic_trace = [], []
    for proposal_attempt in range(1, max_repairs + 2):
        tool_choice = forced_tool or "auto"
        response = model.bind_tools(
            list(TOOLS.values()), tool_choice=tool_choice,
        ).invoke(messages + [step_prompt, *feedback_messages])
        review = critic_review(
            response.tool_calls,
            expected_tool=expected_tool,
            expected_arguments=expected_arguments,
            forced_tool=forced_tool,
        )
        critic_trace.append({
            "proposal_attempt": proposal_attempt,
            "tool_choice": tool_choice,
            "outcome": review.outcome,
            "feedback": review.feedback,
        })
        if review.outcome == "approved":
            return response, review.call, critic_trace
        if review.outcome == "policy_denied":
            raise ToolCallAuthorizationError(review.feedback)
        if proposal_attempt > max_repairs:
            raise ToolCallValidationError(
                f"лимит исправлений исчерпан: {review.feedback}"
            )
        # Отклонённый AIMessage не добавляем: его вызову не соответствует ToolMessage.
        feedback_messages.append(clarification_for(review))
    raise AssertionError("недостижимая ветка")

repair_protocol = {
    "максимум_исправлений": 1,
    "исправимые_ошибки": [
        "неверные аргументы", "преждевременный разрешённый инструмент",
        "неверная форма вызова",
    ],
    "конечные_ошибки": ["инструмент вне списка разрешённых"],
    "правило_истории": "отклонённый AIMessage не добавляется",
}

pprint({
    "протокол_критика_и_исправления": repair_protocol,
})

{'протокол_критика_и_исправления': {'исправимые_ошибки': ['неверные аргументы',
                                                          'преждевременный '
                                                          'разрешённый '
                                                          'инструмент',
                                                          'неверная форма '
                                                          'вызова'],
                                    'конечные_ошибки': ['инструмент вне списка '
                                                        'разрешённых'],
                                    'максимум_исправлений': 1,
                                    'правило_истории': 'отклонённый AIMessage '
                                                       'не добавляется'}}


Ограниченное исправление повторяет обращение к модели только при исправимой ошибке; отказ по правилам завершает цикл, а отклонённый `AIMessage` не загрязняет историю API.

### Один шаг ReAct

Чтобы итоговый цикл читался сверху вниз, один шаг вынесен отдельно. Он получает текущее состояние и проверенные факты, запрашивает проверенное действие, устойчиво читает данные, создаёт `ToolMessage` и возвращает следующее состояние вместе с трассировкой.

In [20]:
class ReactStepResult(NamedTuple):
    next_state: str
    fact_name: str
    fact_value: dict
    history_items: tuple
    trace_item: dict
    model_calls: int

def execute_react_step(
    model, messages, *, state, payment_id, facts, cache, plan,
    force_tool=False,
) -> ReactStepResult:
    """Выполняет один шаг: действие → наблюдение → переход состояния."""
    expected_tool = EXPECTED_TOOL_BY_STATE[state]
    expected_arguments = expected_arguments_for(expected_tool, payment_id, facts)
    visible_plan = remaining_plan(state, plan)
    step_prompt = HumanMessage(content=(
        f"Текущее состояние: {state}. Идентификатор платежа: {payment_id}. "
        f"Оставшийся план: {visible_plan}. Проверенные факты: {facts}. "
        "Выбери следующий инструмент и используй только идентификаторы из этих данных."
    ))
    response, call, critic_trace = request_validated_action(
        model, messages, step_prompt,
        expected_tool=expected_tool,
        expected_arguments=expected_arguments,
        forced_tool=expected_tool if force_tool else None,
    )

    if expected_tool == "get_account":
        resilient = read_or_degrade(
            expected_tool, call["args"],
            lambda args: TOOLS[expected_tool].invoke(args), cache,
        )
    else:
        read = ToolReadResult(
            TOOLS[expected_tool].invoke(call["args"]), 1, False, (),
        )
        resilient = ResilientReadOutcome("ok", read, "continue", None)

    read = resilient.read
    observation = {
        "status": resilient.status, "data": read.value,
        "attempts": read.attempts, "cache_hit": read.cache_hit,
        "backoff_delays": read.backoff_delays,
        "route": resilient.route, "error": resilient.error,
    }
    event = event_from_observation(state, expected_tool, read.value)
    next_state = transition_state(state, event)
    trace_item = {
        "state": state, "tool": expected_tool, "arguments": call["args"],
        "event": event, "tool_call_id": call["id"],
        "observation": observation, "plan_before_step": visible_plan,
        "critic": critic_trace,
    }
    return ReactStepResult(
        next_state, expected_tool, read.value,
        (step_prompt, response, make_tool_message(call, observation)),
        trace_item, len(critic_trace),
    )

pprint({
    "контракт_шага_react": {
        "вход": ["state", "payment_id", "facts", "cache", "plan"],
        "последовательность": [
            "ожидаемые аргументы", "предложение GigaChat", "критик и исправление",
            "устойчивое чтение", "ToolMessage", "переход FSM",
        ],
        "выход": list(ReactStepResult._fields),
    }
})

{'контракт_шага_react': {'вход': ['state',
                                  'payment_id',
                                  'facts',
                                  'cache',
                                  'plan'],
                         'выход': ['next_state',
                                   'fact_name',
                                   'fact_value',
                                   'history_items',
                                   'trace_item',
                                   'model_calls'],
                         'последовательность': ['ожидаемые аргументы',
                                                'предложение GigaChat',
                                                'критик и исправление',
                                                'устойчивое чтение',
                                                'ToolMessage',
                                                'переход FSM']}}


Одна вспомогательная функция связывает план, критик, устойчивое чтение, ToolMessage и переход FSM; внешний цикл лишь накапливает историю и останавливается.

### Цикл выполнения: повторяем шаг до финального состояния

In [21]:
def run_react_agent(
    question: str, *, payment_id: str, force_first_tool: bool = False,
    read_cache: dict | None = None,
):
    """Повторяет ограниченные шаги ReAct, затем получает итоговый ответ."""
    account_service.reset()
    model = build_gigachat()
    state, facts = "проверка_платежа", {}
    cache = {} if read_cache is None else read_cache
    state_path, trace, model_calls = [state], [], 0
    initial_plan = plan_before_calls(payment_id)
    messages = [
        SystemMessage(content=(
            "Ты помощник поддержки. Используй только инструменты, переданные "
            "средой выполнения. После наблюдений дай короткий ответ на русском "
            "без новых действий."
        )),
        HumanMessage(content=question),
    ]

    while state in EXPECTED_TOOL_BY_STATE:
        step = execute_react_step(
            model, messages, state=state, payment_id=payment_id,
            facts=facts, cache=cache, plan=initial_plan,
            force_tool=force_first_tool and not trace,
        )
        facts[step.fact_name] = step.fact_value
        messages.extend(step.history_items)
        trace.append(step.trace_item)
        model_calls += step.model_calls
        state = step.next_state
        state_path.append(state)

    final_response = model.invoke(messages + [HumanMessage(content=(
        "Проверки завершены. Объясни результат простыми словами. "
        "Не предлагай выполнить возврат автоматически."
    ))])
    model_calls += 1
    state = transition_state(state, "ответ_готов")
    state_path.append(state)
    degraded = any(
        step["observation"]["status"] == "degraded" for step in trace
    )
    return {
        "status": "degraded" if degraded else "success",
        "route": "manual_review" if degraded else "agent",
        "model": GIGACHAT_MODEL,
        "model_calls": model_calls, "state_path": state_path,
        "initial_plan": initial_plan, "tool_steps": trace,
        "final_answer": final_response.content,
    }

pprint({
    "цикл_агента": {
        "функция": run_react_agent.__name__,
        "модель": GIGACHAT_MODEL,
        "шаги": "execute_react_step до подготовка_ответа, затем итоговый ответ",
        "результат_при_деградации": {"status": "degraded", "route": "manual_review"},
    }
})

{'цикл_агента': {'модель': 'GigaChat-2-Max',
                 'результат_при_деградации': {'route': 'manual_review',
                                              'status': 'degraded'},
                 'функция': 'run_react_agent',
                 'шаги': 'execute_react_step до подготовка_ответа, затем '
                         'итоговый ответ'}}


Внешний цикл агента повторяет проверяемый шаг ReAct, накапливает трассировку и отражает деградацию в итоговых полях `status` и `route`.

### Вход в приложение: маршрутизация до модели

`run_support_request` сначала отвечает на точный FAQ по заданному правилу или просит уточнение. GigaChat создаётся только для агентной ветки. `payment_id` из текста и явный аргумент не могут молча разойтись.

In [22]:
def run_support_request(
    question: str, *, payment_id: str | None = None,
    force_first_tool: bool = False, read_cache: dict | None = None,
):
    """Обрабатывает точные и неподдерживаемые запросы до создания GigaChat."""
    decision = route_request(question)
    if decision.route != "agent":
        return {
            "status": (
                "needs_clarification"
                if decision.route == "unsupported" else "success"
            ),
            "route": decision.route,
            "label": decision.label, "model": None, "model_calls": 0,
            "state_path": [decision.route], "initial_plan": [],
            "tool_steps": [], "final_answer": decision.answer,
        }
    mentioned_ids = {
        item.upper() for item in re.findall(r"\bP-\d+\b", question, re.IGNORECASE)
    }
    if payment_id is None:
        if len(mentioned_ids) != 1:
            raise ValueError("маршрут агента требует ровно один payment_id в вопросе")
        payment_id = next(iter(mentioned_ids))
    elif mentioned_ids and payment_id.upper() not in mentioned_ids:
        raise ValueError("payment_id не совпадает с идентификатором в вопросе")
    return run_react_agent(
        question, payment_id=payment_id.upper(),
        force_first_tool=force_first_tool, read_cache=read_cache,
    )

try:
    run_support_request("Проверь платёж P-76", payment_id="P-77")
    raise AssertionError("среда выполнения должна отклонить два разных payment_id")
except ValueError as error:
    payment_id_mismatch = str(error)

pprint({
    "среда_выполнения_react": {
        "точка_входа": run_support_request.__name__,
        "модель": GIGACHAT_MODEL,
        "маршруты": {
            "точный": "0 обращений к модели",
            "дубликат": "3 инструмента и итоговый ответ",
            "обычный": "1 инструмент и итоговый ответ",
        },
        "использует_предыдущие_функции": [
            "route_request", "plan_before_calls", "critic_review",
            "execute_react_step",
            "transition_state", "event_from_observation", "read_or_degrade",
            "read_with_retry_cache", "make_tool_message",
        ],
    },
    "пример_неподдерживаемого_запроса": run_support_request("У меня проблема с оплатой"),
    "несовпадение_payment_id": payment_id_mismatch,
})

{'несовпадение_payment_id': 'payment_id не совпадает с идентификатором в '
                            'вопросе',
 'пример_неподдерживаемого_запроса': {'final_answer': 'Уточните тип проблемы и '
                                                      'идентификатор платежа '
                                                      'вида P-123.',
                                      'initial_plan': [],
                                      'label': 'нужно_уточнение',
                                      'model': None,
                                      'model_calls': 0,
                                      'route': 'unsupported',
                                      'state_path': ['unsupported'],
                                      'status': 'needs_clarification',
                                      'tool_steps': []},
 'среда_выполнения_react': {'использует_предыдущие_функции': ['route_request',
                                                              'plan_before_calls',
   

Вход в приложение соединяет классификатор и цикл агента: FAQ и неподдерживаемый запрос не создают модель, а payment_id из вопроса проверяется против явного аргумента.

### Три сетевых запуска и один маршрут без модели

In [23]:
def expected_model_calls(run: dict) -> int:
    """Считает предложения, исправления и одно обращение за итоговым ответом."""
    return 1 + sum(len(step["critic"]) for step in run["tool_steps"])

# Один кэш живёт в рамках сессии поддержки, а не вечно внутри функции.
shared_read_cache = {}

# Длинная ветка: три инструмента и итоговый ответ.
duplicate_run = run_support_request(
    "У клиента двойное списание по платежу P-77. Проверь все необходимые факты.",
    force_first_tool=True, read_cache=shared_read_cache,
)
assert duplicate_run["status"] == "success"
assert duplicate_run["model"] == "GigaChat-2-Max"
assert duplicate_run["model_calls"] == expected_model_calls(duplicate_run)
assert 4 <= duplicate_run["model_calls"] <= 7
assert duplicate_run["state_path"] == [
    "проверка_платежа", "проверка_аккаунта", "проверка_правил",
    "подготовка_ответа", "завершено",
]

# Точный повтор в той же сессии: чтение аккаунта не обращается к службе данных.
replay_run = run_support_request(
    "Повторно проверь двойное списание P-77 в этой же сессии.",
    read_cache=shared_read_cache,
)
assert replay_run["status"] == "success"
assert replay_run["model_calls"] == expected_model_calls(replay_run)
assert 4 <= replay_run["model_calls"] <= 7
replay_account_step = next(
    step for step in replay_run["tool_steps"] if step["tool"] == "get_account"
)
assert replay_account_step["observation"]["cache_hit"] is True
assert replay_account_step["observation"]["attempts"] == 0

# Короткая ветка: наблюдение первого инструмента показывает, что дубликата нет.
regular_run = run_support_request(
    "Проверь, является ли платёж P-76 двойным списанием.",
)
assert regular_run["status"] == "success"
assert regular_run["model_calls"] == expected_model_calls(regular_run)
assert 2 <= regular_run["model_calls"] <= 3
assert regular_run["state_path"] == [
    "проверка_платежа", "подготовка_ответа", "завершено",
]

# Точная ветка: клиент GigaChat даже не создаётся.
deterministic_run = run_support_request("Как сменить пароль?")
assert deterministic_run["route"] == "deterministic"
assert deterministic_run["model_calls"] == 0

pprint({
    "результаты_react_агента": {
        "источник": "GigaChat",
        "двойное_списание": duplicate_run,
        "повтор_в_том_же_сеансе": replay_run,
        "обычный_платёж": regular_run,
        "точный_faq": deterministic_run,
    }
})

{'результаты_react_агента': {'двойное_списание': {'final_answer': 'Произошло '
                                                                  'двойное '
                                                                  'списание '
                                                                  'средств по '
                                                                  'платежу '
                                                                  'P-77. '
                                                                  'Аккаунт '
                                                                  'клиента '
                                                                  'активен, '
                                                                  'поэтому '
                                                                  'случай '
                                                                  'подлежит '
                                                                  'ручн

Сетевые сценарии показывают принудительный и автоматический Function Calling, исправление критиком, один или три шага с инструментами, попадание в кэш и точный FAQ с `model_calls=0`.

### Сравниваем стоимость четырёх маршрутов

Три трассировки получены при обращении к GigaChat-2-Max в одной среде выполнения, а точный FAQ обработан до создания модели. Повтор P-77 использует общий кэш сеанса для чтения аккаунта. Сравним количество обращений к модели, выбранные инструменты, попадания в кэш и путь по состояниям.

In [24]:
def compare_react_runs(runs: dict[str, dict]) -> list[dict]:
    """Сводит трассировки сетевых запусков в компактную таблицу."""
    return [
        {
            "scenario": scenario,
            "route": run["route"],
            "model_calls": run["model_calls"],
            "tools": [step["tool"] for step in run["tool_steps"]],
            "events": [step["event"] for step in run["tool_steps"]],
            "cache_hits": [
                step["tool"] for step in run["tool_steps"]
                if step["observation"]["cache_hit"]
            ],
            "state_path": run["state_path"],
            "final_answer": run["final_answer"],
        }
        for scenario, run in runs.items()
    ]

trace_comparison = compare_react_runs({
    "двойное_списание": duplicate_run,
    "повтор_с_кэшем": replay_run,
    "обычный_платёж": regular_run,
    "точный_faq": deterministic_run,
})
assert len(trace_comparison[0]["tools"]) == 3
assert len(trace_comparison[1]["tools"]) == 3
assert len(trace_comparison[2]["tools"]) == 1
assert len(trace_comparison[3]["tools"]) == 0
assert trace_comparison[3]["model_calls"] == 0

pprint({
    "сравнение_трассировок": trace_comparison,
    "различие": (
        "3 инструмента, точный повтор из кэша, 1 инструмент и 0 обращений "
        "к модели: наблюдения, кэш и маршрутизация меняют стоимость"
    ),
})

{'различие': '3 инструмента, точный повтор из кэша, 1 инструмент и 0 обращений '
             'к модели: наблюдения, кэш и маршрутизация меняют стоимость',
 'сравнение_трассировок': [{'cache_hits': [],
                            'events': ['дубликат_найден',
                                       'аккаунт_активен',
                                       'правило_получено'],
                            'final_answer': 'Произошло двойное списание '
                                            'средств по платежу P-77. Аккаунт '
                                            'клиента активен, поэтому случай '
                                            'подлежит ручной проверке для '
                                            'возможного возврата средств. '
                                            'Решение должно быть принято в '
                                            'течение 24 часов.',
                            'model_calls': 5,
                            'route': 'agent',
   

Сводная трассировка показывает фактические обращения к модели, ветвление по наблюдению, кэш сеанса и нулевую стоимость точного маршрута.

### Как механизмы соединены

`классификатор → план → вызов GigaChat → критик/исправление → проверка FSM → Python-функция → повтор/кэш → ToolMessage → новое состояние`

Цикл повторяет эту последовательность столько раз, сколько требует текущий сценарий. Обычный платёж использует один вызов инструмента, двойное списание — три. После последнего наблюдения GigaChat пишет итоговый ответ.

Важно разделять роли:

- модель в режиме `auto` предлагает вызов и формулирует ответ;
- среда выполнения строит план, разрешает ровно один инструмент текущего состояния и по наблюдению выбирает ветку;
- критик может один раз вернуть модели конкретное уточнение;
- Python-функция получает данные;
- повторные попытки, задержка, кэш и деградация относятся к среде выполнения, а не к модели;
- компенсация принадлежит доверенной части бизнес-приложения и не передаётся модели как инструмент.

Это **ограниченный ReAct**, а не полностью автономный агент: GigaChat предлагает действие, но FSM ограничивает разрешённый шаг. Если бы Python заранее задавал весь неизменный маршрут без ветвления по наблюдениям, это был бы фиксированный рабочий процесс.

## Проверка готовности

После выполнения ноутбука проверьте себя:

- можете своими словами объяснить цикл `вызов инструмента → наблюдение → следующий шаг`;
- принудительный выбор проходит ту же проверку схемы и списка разрешённых инструментов, что и режим `auto`;
- параллельная группа принимает только два независимых вызова для чтения и сохраняет оба `tool_call_id`;
- наблюдаемый план сокращается после каждого наблюдения;
- критик различает `approved`, `repair` и `policy_denied`;
- `transition_state` отклоняет недопустимый переход;
- при первом чтении аккаунта имитация API отвечает ошибкой `429`, а повторная попытка завершается успешно;
- повторное точно такое же чтение приходит из кэша с `attempts=0`;
- повторный сетевой запуск P-77 в том же сеансе показывает `cache_hit=True` на `get_account`;
- две последовательные временные ошибки дают типизированную деградацию `manual_review`;
- цепочка операций с ошибкой завершается `compensated` и `net_effect=None`;
- точный FAQ проходит ветку без модели с `model_calls=0`;
- неверные имя инструмента, аргументы, `call_id` и размер группы отклоняются до Python-вызова;
- наблюдение платежа выбирает одну из веток FSM;
- трассировка двойного списания проходит `проверка_платежа → проверка_аккаунта → проверка_правил → подготовка_ответа → завершено`;
- трассировка обычного платежа проходит `проверка_платежа → подготовка_ответа → завершено`;
- итоговый текст создан GigaChat только после всех необходимых проверенных наблюдений.

Для сетевого запуска откройте ноутбук, выполните ячейки по порядку и запустите последнюю ячейку после заполнения `.env`.

## Итоги

Мы прошли путь от отдельного Function Calling до ветвящегося ReAct-агента:

- схемы параметров, принудительный выбор и правила параллелизма защищают границу между API модели и средой выполнения;
- наблюдаемый план, критик и ограниченное исправление управляют предложениями модели;
- FSM фиксирует состояние и инварианты шага;
- повторные попытки, нарастающая задержка, кэш и деградация обрабатывают ошибки чтения;
- компенсация показывает восстановление после частичного побочного эффекта;
- классификатор и ветка без модели экономят обращения к GigaChat.

Главный результат — не универсальная платформа, а наблюдаемая среда выполнения с тремя маршрутами: длинным, коротким и заданным точным правилом. Её можно прочитать сверху вниз и проверить по трассировке.

### Чему вы научились

Вы умеете связать до трёх зависимых вызовов инструментов, выбирать ветку по наблюдению, исправлять ответ модели в пределах заданного числа попыток, управлять FSM и отделять повтор запроса к модели от повтора запроса к службе данных и компенсации.

В третьем занятии этот один разросшийся агент будет разделён: предметные правила станут навыком, а узкая проверка правила — задачей дочернего агента с собственными инструментами и бюджетом.

### Контрольные вопросы

1. Почему принудительный выбор не отменяет проверку среды выполнения?
2. Когда два вызова только для чтения всё равно нельзя выполнять параллельно?
3. Чем наблюдаемый план отличается от скрытой цепочки рассуждений (chain-of-thought)?
4. Когда критик просит исправить вызов, а когда завершает запуск?
5. Кто меняет состояние: GigaChat или Python-приложение?
6. Чем повторная попытка отличается от деградации и компенсации?
7. Почему точный FAQ должен обходить модель?
8. Почему безопасно кэшировать только успешное чтение?